In [0]:
spark.sql("USE CATALOG dbw_fleet_telemetry_dev")
# Silver notebook configuration

storage_account   = "stfleettelemetryalvin"
storage_key       = "YOUR_STORAGE_ACCOUNT_KEY"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

CHECKPOINT_PATH = f"abfss://bronze@{storage_account}.dfs.core.windows.net/_checkpoints/fleet_silver"

print("Config loaded")

# Verify Bronze table has data
spark.sql("SELECT COUNT(*) as total_rows FROM bronze.raw_telemetry").show()

Config loaded
+----------+
|total_rows|
+----------+
|    163030|
+----------+



In [0]:
from pyspark.sql import functions as F

# Load static vehicle reference dimension
vehicle_ref = (
    spark.read
    .table("reference.vehicle_dimension")
    .select("vehicle_id", "driver_name", "route_id", 
            "route_name", "vehicle_type", "max_speed_kmh", "home_depot")
)

vehicle_ref.cache()
print(f"Vehicle reference loaded: {vehicle_ref.count()} rows")

# Read Bronze as stream with watermark
bronze_stream = (
    spark.readStream
    .format("delta")
    .table("bronze.raw_telemetry")
    .withWatermark("event_time", "1 minutes")
)

print("Bronze stream with watermark defined")

Vehicle reference loaded: 10 rows
Bronze stream with watermark defined


In [0]:
windowed_stats = (
    bronze_stream
    .groupBy(
        F.window("event_time", "1 minute"),
        "vehicle_id",
        "route_id",
    )
    .agg(
        F.avg("speed_kmh").alias("avg_speed_kmh"),
        F.max("speed_kmh").alias("max_speed_kmh"),
        F.avg("engine_temp_c").alias("avg_engine_temp_c"),
        F.max("engine_temp_c").alias("max_engine_temp_c"),
        F.min("fuel_pct").alias("min_fuel_pct"),
        F.count("*").alias("event_count"),
        F.sum(
            F.when(F.col("alert_type") != "normal", 1).otherwise(0)
        ).alias("anomaly_event_count"),
        F.last("alert_type").alias("alert_type"),
        F.last("severity").alias("severity"),
        F.last("lat").alias("last_lat"),
        F.last("lon").alias("last_lon"),
    )
    .withColumn("window_start", F.col("window.start"))
    .withColumn("window_end",   F.col("window.end"))
    .drop("window")
)

print("Windowed aggregation defined")
windowed_stats.printSchema()

Windowed aggregation defined
root
 |-- vehicle_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- avg_speed_kmh: double (nullable = true)
 |-- max_speed_kmh: float (nullable = true)
 |-- avg_engine_temp_c: double (nullable = true)
 |-- max_engine_temp_c: float (nullable = true)
 |-- min_fuel_pct: float (nullable = true)
 |-- event_count: long (nullable = false)
 |-- anomaly_event_count: long (nullable = true)
 |-- alert_type: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- last_lat: double (nullable = true)
 |-- last_lon: double (nullable = true)
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)



In [0]:
# Stream-static join — enrich with vehicle dimension
vehicle_ref_renamed = vehicle_ref.withColumnRenamed("max_speed_kmh", "rated_max_speed")

enriched_stream = (
    windowed_stats
    .join(
        F.broadcast(vehicle_ref_renamed),
        on="vehicle_id",
        how="left"
    )
    .withColumn(
        "speed_pct_of_max",
        F.round(F.col("avg_speed_kmh") / F.col("rated_max_speed") * 100, 1)
    )
    .withColumn("ingestion_time", F.current_timestamp())
)

print("Stream-static join defined")
enriched_stream.printSchema()

Stream-static join defined
root
 |-- vehicle_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- avg_speed_kmh: double (nullable = true)
 |-- max_speed_kmh: float (nullable = true)
 |-- avg_engine_temp_c: double (nullable = true)
 |-- max_engine_temp_c: float (nullable = true)
 |-- min_fuel_pct: float (nullable = true)
 |-- event_count: long (nullable = false)
 |-- anomaly_event_count: long (nullable = true)
 |-- alert_type: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- last_lat: double (nullable = true)
 |-- last_lon: double (nullable = true)
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- driver_name: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- route_name: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- rated_max_speed: long (nullable = true)
 |-- home_depot: string (nullable = true)
 |-- speed_pct_of_max: double (nullable = true)
 |-- ing

In [0]:
# Drop duplicate route_id from reference side
enriched_stream = enriched_stream.drop(vehicle_ref_renamed.route_id)

print("Duplicate dropped")
enriched_stream.printSchema()

Duplicate dropped
root
 |-- vehicle_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- avg_speed_kmh: double (nullable = true)
 |-- max_speed_kmh: float (nullable = true)
 |-- avg_engine_temp_c: double (nullable = true)
 |-- max_engine_temp_c: float (nullable = true)
 |-- min_fuel_pct: float (nullable = true)
 |-- event_count: long (nullable = false)
 |-- anomaly_event_count: long (nullable = true)
 |-- alert_type: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- last_lat: double (nullable = true)
 |-- last_lon: double (nullable = true)
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- driver_name: string (nullable = true)
 |-- route_name: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- rated_max_speed: long (nullable = true)
 |-- home_depot: string (nullable = true)
 |-- speed_pct_of_max: double (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)



In [0]:
silver_query = (
    enriched_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime="30 seconds")
    .toTable("silver.vehicle_window_stats")
)

print(f"Silver stream started")
print(f"Query ID: {silver_query.id}")

In [0]:
spark.sql("""
    SELECT alert_type, severity, COUNT(*) as count
    FROM silver.vehicle_window_stats
    GROUP BY alert_type, severity
    ORDER BY count DESC
""").show()

+---------------+--------+-----+
|     alert_type|severity|count|
+---------------+--------+-----+
|         normal|  NORMAL|  202|
|   speed_breach|  MEDIUM|   37|
|engine_overheat|CRITICAL|   25|
|  fuel_critical|    HIGH|   22|
|route_deviation|  MEDIUM|    4|
+---------------+--------+-----+

